# Feature Wrangling

In this lecture, we will refresh how to properly perform **feature wrangling**. This is a fundamental process in Machine Learning.

By the end, you should be able to:
- Distinguish fixed from fitted transformations;
- Handle missingness without contaminating the holdout set;
- explain when scaling matters and how it should be fitted;
- distinguish engineering, selection, and dimensionality reduction;
- diagnose leakage in feature selection;
- use a `scikit-learn` pipeline for a clean train/holdout workflow.


For this walkthrough lecture, we will use the following synthetic data:

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
rng = np.random.default_rng(42)

n = 1200

# -----------------------------
# Numerical features
# -----------------------------
age = rng.normal(40, 12, n).clip(18, 80)
income = rng.lognormal(10.7, 0.75, n)
credit_score = rng.normal(680, 65, n).clip(300, 850)
transactions = rng.poisson(25, n)
debt_ratio = rng.beta(2, 6, n)
noise_1 = rng.normal(size=n)
noise_2 = rng.normal(size=n)

# -----------------------------
# Nominal categorical feature
# No natural ordering
# -----------------------------
region = rng.choice(
    ["North", "South", "East", "West"],
    size=n,
    p=[0.25, 0.30, 0.25, 0.20]
)

region_effect = np.select(
    [
        region == "North",
        region == "South",
        region == "East",
        region == "West",
    ],
    [0.35, -0.20, 0.10, 0.00]
)

# -----------------------------
# Ordinal categorical feature
# Low < Medium < High
# -----------------------------
engagement_level = rng.choice(
    ["Low", "Medium", "High"],
    size=n,
    p=[0.30, 0.45, 0.25]
)

engagement_effect = np.select(
    [
        engagement_level == "Low",
        engagement_level == "Medium",
        engagement_level == "High",
    ],
    [-0.30, 0.00, 0.35]
)

# -----------------------------
# Generate binary target
# -----------------------------
eta = (
    -2
    + 0.025 * (age - 40)
    + 0.000025 * (income - np.median(income))
    + 0.010 * (credit_score - 680)
    - 3 * debt_ratio
    + 0.04 * transactions
    + region_effect
    + engagement_effect
)

p = 1 / (1 + np.exp(-eta))
target = rng.binomial(1, p)

# -----------------------------
# DataFrame
# -----------------------------
df = pd.DataFrame({
    "age": age,
    "income": income,
    "credit_score": credit_score,
    "transactions": transactions,
    "debt_ratio": debt_ratio,
    "noise_1": noise_1,
    "noise_2": noise_2,
    "region": region,
    "engagement_level": engagement_level,
    "target": target,
})

# Explicitly represent the ordinal variable as ordered
df["engagement_level"] = pd.Categorical(
    df["engagement_level"],
    categories=["Low", "Medium", "High"],
    ordered=True
)

# -----------------------------
# Introduce missingness
# -----------------------------
df.loc[rng.random(n) < 0.12, "credit_score"] = np.nan
df.loc[
    rng.random(n) < (0.04 + 0.15 * (age < 28)),
    "income"
] = np.nan

# Feature groups for later preprocessing
numeric_features = [
    "age",
    "income",
    "credit_score",
    "transactions",
    "debt_ratio",
    "noise_1",
    "noise_2",
]

nominal_features = ["region"]
ordinal_features = ["engagement_level"]

df.head()

In [ ]:
# Separate the input data (X) from the outcome (y)
X_transform = df.drop(columns="target")
y_transform = df["target"]

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler, PowerTransformer, KBinsDiscretizer
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, f_classif, RFE
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

rng = np.random.default_rng(42)

## 0. Keep this rule firmly in your mind!!

$$
\boxed{\text{Split first. Learn from train. Apply to holdout.}}
$$

Every section asks the same question:

> **What is being learned from the data, and which observations are allowed to influence that learning?**


## 1. The train/holdout contract

$$
\boxed{\text{The holdout set is for evaluation, not decision-making.}}
$$

```text
RAW DATA
   |
   v
TRAIN / HOLDOUT SPLIT
   |
   +------ TRAIN ------> learn preprocessing ------> fit model
   |
   +------ HOLDOUT  ------> apply learned preprocessing ------> evaluate
```

If a preprocessing step **estimates, chooses, or learns** something from the data, that learning must belong on the training side.

In `scikit-learn` language:

```python
transformer.fit(X_train)
X_train_new = transformer.transform(X_train)
X_holdout_new  = transformer.transform(X_holdout)
```

The holdout set may be transformed, but should never teach us the transformation.

### Starting question

For each operation below, which do you think **learns something from the observed data**?

1. `np.log1p(x)`
2. median imputation
3. standardization
4. selecting the 10 variables most associated with `y`

## 2. Transformations: fixed vs learned

Transformations may compress extremes, reduce skew, improve linearity, stabilize variance, or encode domain knowledge.

The important distinction today is:

$$
\boxed{\text{fixed transformation} \neq \text{fitted transformation}}
$$

The most important thing to ask is: Does it learn anything from the observed sample? If it does, where should that learning happen?


### 2.1 Transforming numeric data

Typical examples here include:

- **Log / square-root transforms** — compress right tails and multiplicative effects.
- **Power transforms** — e.g. Yeo–Johnson; estimate a transformation parameter from data.
- **Clipping / winsorization** — cap extreme values.

In [ ]:
# Use this function to produce a train/holdout split
from sklearn.model_selection import train_test_split

In [ ]:
# Compare a fixed log transform with a fitted Yeo-Johnson transform
from sklearn.preprocessing import PowerTransformer

X_train_transform, X_holdout_transform, y_train_transform, y_holdout_transform = train_test_split(
    X_transform, y_transform, test_size=.30, random_state=42, stratify=y_transform
)

income_train = X_train_transform[["income"]].dropna()


pt = None # HERE COMES CODE: Define a Yeo-Johnson transformation with standardize=False
income_yj = None # HERE COMES CODE: Use the previous object to fit and transform on income_train

print("Learned Yeo-Johnson lambda:", round(float(pt.lambdas_[0]), 3))

plt.figure(figsize=(7, 4))
plt.hist(np.log1p(income_train["income"]), bins=35, alpha=0.7, label="log1p")
plt.hist(income_yj.ravel(), bins=35, alpha=0.5, label="Yeo-Johnson")
plt.legend()
plt.title("Fixed vs fitted transformation")
plt.show()

Look at the effect of the logarithmic transformation in more detail:

In [ ]:
# Just focus on income
income_obs = df["income"].dropna()

plt.figure(figsize=(7,4))
plt.hist(income_obs, bins=40)
plt.xlabel("income"); plt.ylabel("count"); plt.title("Original income")
plt.show()

plt.figure(figsize=(7,4))
plt.hist(np.log1p(income_obs), bins=40)
plt.xlabel("log(1 + income)"); plt.ylabel("count"); plt.title("Log-transformed income")
plt.show()

### Binning as another fitted transformation

If cut points come from the data, they are learned parameters too.

For example, quantile-based bins learned on the training set:

```python
binner = KBinsDiscretizer(n_bins=4, strategy="quantile")
binner.fit(X_train[["income"]])
```

The same cut points must then be applied to the holdout set.

In [ ]:
# HERE COMES CODE: Import KBinsDiscretizer

# Remember: in scikit-learn, an estimator is an instance of a class with methods such as fit and transform
binner = None # HERE COMES CODE: Instantiate KBinsDiscretizer with n_bins=4 and strategy="quantile"
# HERE COMES CODE: Fit to the age column in X_train_transform

In [ ]:
# Remember: these models will also have attributes
print(binner.n_bins_) # The number of bins
print(binner.bin_edges_) # The bin edges

In [ ]:
# We use the learned bins to transform our data
X_train_binner = None # HERE COMES CODE: Transform the training set using the learned transformation and store in this new variable
X_train_binner

In [ ]:
# By default, this returns a CSR matrix, which has a method for converting the data to a dense array
X_train_binner.toarray()

### In-class activity 

Here we will consider two different transformations: one that generates polynomial features and a second one that transforms features using quantile information.

- What are polynomial features? They are all combinations of the features (including interaction terms) up to a certain specified degree. For example, if an input sample is two-dimensional and of the form [a, b], the degree-2 polynomial features are [1, a, b, a^2, ab, b^2].

Let's use their implementations in `scikit-learn` for this.

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
PolynomialFeatures(degree=2).fit_transform(X_transform[["age"]])

Now we add this new set of features to the existing data, creating a new feature matrix:

In [ ]:
X_transform_polynomial = np.column_stack((X_transform, PolynomialFeatures(degree=2).fit_transform(X_transform[["age"]])))
X_transform_polynomial.shape

- Let's do the same, but now using quantile information. For this, we will use `scikit-learn`. This will first estimate the cumulative distribution function of a feature, which is used to map the original values to a uniform distribution. The obtained values are then mapped to the desired output distribution using the associated quantile function. Feature values in new or unseen data that fall below or above the fitted range will be mapped to the bounds of the output distribution. 

In [ ]:
from sklearn.preprocessing import QuantileTransformer

X_transform_quantile = np.column_stack((X_transform, QuantileTransformer(n_quantiles=10).fit_transform(X_transform[["age"]])))
X_transform_quantile.shape

**Question**: Now suppose we split the data into training and holdout sets and perform predictive analysis. In which of these two cases would we be wrong?

A. 
```python
train_test_split(X_polynomial)
```

B. 
```python
train_test_split(X_quantile)
```


### 2.2 Transforming categorical data

Most ML algorithms can't handle features in the form of strings. 

They typically need to be encoded in numbers. We could use `OrdinalEncoder` from `scikit-learn` for this:

In [ ]:
from sklearn.preprocessing import OrdinalEncoder
OrdinalEncoder(categories=[['South', 'West', 'North', 'East']]).fit_transform(X_transform[["region"]])

In [ ]:
OrdinalEncoder(categories=[['Low', 'Medium', 'High']]).fit_transform(X_transform[["engagement_level"]])

**Question**: Do you see anything wrong with any of these?

Alternatively, one-hot encoding creates a binary column for each category. This is particularly useful for categorical data whose categories do not have a natural order. This transformation can be performed with `OneHotEncoder` in `scikit-learn`.

In [ ]:
from sklearn.preprocessing import OneHotEncoder

OneHotEncoder(sparse_output=False).fit_transform(X_transform[["region"]])

We can put both together very easily using `ColumnTransformer` from `scikit-learn`.

In [ ]:
# REMEMBER: (1) import, (2) define the object, and (3) use its methods (e.g., fit and fit_transform)

# HERE COMES CODE: Import ColumnTransformer from compose, and OrdinalEncoder and OneHotEncoder from preprocessing

encoder_columns = None # HERE COMES CODE: Instantiate a ColumnTransformer object in which OneHotEncoder acts on region and OrdinalEncoder 
                       #                  acts on engagement_level. Let it pass through the remaining features

 # HERE COMES CODE: Apply encoder_columns to df[["region", "engagement_level"]] to show that it worked

## 3. Imputation of missing data

We can see that our data contains missing information.

In [ ]:
# Let's work with only the numeric data!
df_missing = df.drop(columns=["target", "region", "engagement_level"])
df_missing.info()

Let's impute this information!

Common strategies for this include: 

- Deletion: Simple, but can waste information and may bias the sample if missingness is systematic.

In [ ]:
# Look how much of the sample we lost!
df_missing.dropna().shape

- Mean imputation: Fast and simple, but sensitive to skew and outliers. We can use `scikit-learn` for this imputation:

In [ ]:
from sklearn.impute import SimpleImputer
SimpleImputer(strategy="mean").fit_transform(df_missing)

- Median imputation: Also fast and simple, and more robust for skewed numerical features. We can use `scikit-learn` for this imputation:

In [ ]:
# HERE COMES CODE: Import SimpleImputer from impute
# HERE COMES CODE: Impute as above, but now by median

- Model-based, using methods such as kNN or iterative imputation. We can use `scikit-learn` for this imputation too:

In [ ]:
from sklearn.impute import KNNImputer

KNNImputer().fit_transform(df.drop(columns=["target", "region", "engagement_level"])).shape

Which one should we choose? Sometimes it depends on the data we are dealing with. 

For example, let's compare mean vs median imputation for the skewed income variable:

In [ ]:
mean_imp = SimpleImputer(strategy="mean")
median_imp = SimpleImputer(strategy="median")

mean_imp.fit(df[["income"]])
median_imp.fit(df[["income"]])

print("Training mean used for imputation:  ", round(mean_imp.statistics_[0], 2))
print("Training median used for imputation:", round(median_imp.statistics_[0], 2))

In [ ]:
# Remember what income looked like:
income_obs = df["income"].dropna()

fig, axs = plt.subplots(figsize=(7,4), nrows=3, sharex=True)
axs[0].hist(income_obs, bins=40)
axs[0].set_title("Original")

axs[1].hist(mean_imp.transform(df[["income"]]), bins=40)
axs[1].set_title("After mean inputation")

axs[2].hist(median_imp.transform(df[["income"]]), bins=40)
axs[2].set_title("After median inputation")

axs[2].set_xlabel("income")
plt.tight_layout()
plt.show()


Since `income` is strongly right-skewed, the mean and median differ substantially.

That does not make median universally “better”; it means the choice should reflect the variable's distribution, robustness requirements, and model.

In [ ]:
# Missingness indicator with scikit-learn
indicator_imp = SimpleImputer(strategy="median", add_indicator=True)
indicator_imp.fit(df[["income", "credit_score"]])

transformed = indicator_imp.transform(df[["income", "credit_score"]])

print("Original columns:", 2)
print("Transformed columns:", transformed.shape[1])
print("Indicator features added for original column indices:",
      indicator_imp.indicator_.features_)

### Exercise: 

**Q**: Why is this median imputation of Income wrong?

```python
X["income"] = X["income"].fillna(median)
X_train, X_holdout = train_test_split(X)
```

Then do it the proper way:

In [ ]:
# YOUR CODE HERE

## 4. Scaling: same rule, different parameters

Sometimes we need to scale our features because some models are sensitive to scaling. Which ones? For example:

- k-NN / k-means
- SVMs
- PCA
- neural networks
- regularized linear/logistic regression

These are usually less sensitive:

- decision trees
- random forests
- gradient-boosted trees

Some examples of scaling operations include:

- **Standard scaling**

$$
z = \frac{x-\mu}{\sigma}
$$

Useful default for many distance-based and regularized models. Found in `scikit-learn` as `preprocessing.StandardScaler`.

- **Min-max scaling**

$$
x' = \frac{x-x_{\min}}{x_{\max}-x_{\min}}
$$

Maps training values approximately into \([0,1]\).  Found in `scikit-learn` as `preprocessing.MinMaxScaler`.

- **Robust scaling**

$$
x' = \frac{x-\mathrm{median}(x)}{\mathrm{IQR}(x)}
$$

Less sensitive to extreme values. Found in `scikit-learn` as `preprocessing.RobustScaler`.


**Question**: What parameters are learned in each of these scaling operations?

In [ ]:
# Compare the three scaling approaches on income
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
income_train = X_train[["income"]]
income_train_imp = SimpleImputer(strategy="median").fit_transform(income_train)

scalers = {
    "standard": StandardScaler(),
    "minmax": MinMaxScaler(),
    "robust": RobustScaler()
}

comparison = pd.DataFrame({"original": income_train_imp[:8, 0]})

for name, sc in scalers.items():
    comparison[name] = sc.fit_transform(income_train_imp)[:8, 0]

comparison

The numerical outputs differ because the three methods encode scale differently.

The more important modeling question is:

> **Does the downstream algorithm care about feature scale at all?**

If not, choosing among scalers may be unnecessary.

### Exercise

Suppose `StandardScaler` is fitted on `X_train`.

- Which statement should be true afterward?

    **A.** Train and holdout both have exactly mean 0.  
    **B.** Train has approximately mean 0; holdout need not.  
    **C.** Holdout has mean 0; train need not.  
    **D.** Neither should have mean 0.

- Also decide: Should we fit a *second* scaler on the holdout set so that the holdout features also have mean 0?

- Would this be different if we used a different scaling operation?


YOUR RESPONSE HERE AS MARKDOWN

### Exercise

Suppose a feature contains several extreme outliers.

Which scaler would you expect to be least influenced by those outliers?

- `StandardScaler`
- `MinMaxScaler`
- `RobustScaler`

And does that automatically mean it is the best choice?

YOUR RESPONSE HERE AS MARKDOWN

## 5. Feature selection

Feature selection tries to keep a subset of existing predictors.


Selection is especially tempting when the number of features ($m$) is greater than the number of observations ($N$).

$$
m \gg N.
$$

That was common in early gene-expression and brain-imaging classification: a few dozen patients, but thousands of candidate genes or voxels. 

There are three broad feature-selection families:

- **Filter methods**

Score variables independently of the final predictive model.

Examples:
- correlation;
- ANOVA / F-statistics;
- chi-square;
- mutual information.

Fast and easy to interpret, but can miss variables that are weak individually and useful jointly.

This implementation can be found in `feature_selection.SelectKBest` in `scikit-learn`:

```python
SelectKBest(score_func=f_classif, k=20)
```

- **Wrapper methods**

Use predictive performance to evaluate subsets.

Examples:
- forward selection;
- backward elimination;
- recursive feature elimination (RFE).

These implementations can also be found in `feature_selection` in `scikit-learn`:

```python
RFE(LogisticRegression(), n_features_to_select=20)
```

Potentially more computationally expensive because the model is repeatedly refit.

- **Embedded methods**

Selection occurs during model fitting.

Examples:
- L1-regularized linear/logistic regression;
- some tree-based methods.

For example, with L1 regularization some coefficients can be shrunk exactly to zero.

These implementations can again be found in `feature_selection` in `scikit-learn`. For example:

```python
SelectFromModel(LogisticRegression(), threshold=0)
```

In [ ]:
# Small illustration
from sklearn.feature_selection import SelectKBest, RFE, SelectFromModel
demo_features = [
    "age", "income", "credit_score", "transactions",
    "debt_ratio", "noise_1", "noise_2"
]

X_demo = df[demo_features]
y_demo = df["target"]

# Impute + scale

X_demo_prep = StandardScaler().fit_transform(SimpleImputer(strategy="median").fit_transform(X_demo))

# Filter
filter_sel = SelectKBest(score_func=f_classif, k=4)
filter_sel.fit(X_demo_prep, y_demo)

# Wrapper
rfe_sel = RFE(
    LogisticRegression(max_iter=2000),
    n_features_to_select=4
)
rfe_sel.fit(X_demo_prep, y_demo)

# Embedded: L1 logistic regression
l1_model = SelectFromModel(LogisticRegression(penalty="l1", solver="liblinear", C=0.1, max_iter=2000), 
                           threshold=1e-6)

l1_model.fit(X_demo_prep, y_demo)

selection_summary = pd.DataFrame({
    "feature": demo_features,
    "filter_selected": filter_sel.get_support(),
    "rfe_selected": rfe_sel.get_support(),
    "l1_nonzero": l1_model.get_support()
})

selection_summary

### Exercise

Match the method to the description:

1. `SelectKBest`
2. RFE
3. L1-regularized logistic regression

Descriptions:

A. Selection happens as part of fitting the predictive model.  
B. Variables are scored before fitting the final predictive model.  
C. The model is repeatedly fitted while candidate features are removed.


### Now look at this impossible prediction problem:

Let's generate:
- \(N=80\) observations;
- \(m=5000\) predictors;
- a completely random binary target.

By construction,

$$
X \perp y.
$$

There is no signal. Honest holdout accuracy should be around chance.

In [ ]:
rng_fs = np.random.default_rng(12345)
N, m = 80, 5000

X_noise = rng_fs.normal(size=(N, m))
y_noise = rng_fs.integers(0, 2, size=N)

print("X shape:", X_noise.shape)
print("class counts:", np.bincount(y_noise))

### Exercise

#### Your turn to make a prediction (1 minute)

Before running the next cells:

- What holdout accuracy would you expect from an honest procedure?
- Approximately 50%? 60%? 80%?
- Why?

Remember:

$$
X \perp y
$$

There is literally no predictive signal.

#### Stop before running the following (2 minutes)

The next cell selects the 20 “best” variables **before** the train/holdout split.

Predict the resulting holdout accuracy.

More importantly, explain:

> How can holdout observations influence the model even though the classifier itself has never been fitted on them?

Run the cell only after committing to an answer.

In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression

selector_wrong = SelectKBest(score_func=f_classif, k=20)
X_selected_wrong = selector_wrong.fit_transform(X_noise, y_noise)

X_train_w, X_holdout_w, y_train_w, y_holdout_w = train_test_split(
    X_selected_wrong, y_noise, test_size=.35,
    random_state=7, stratify=y_noise
)

model_wrong = LogisticRegression(max_iter=3000)
model_wrong.fit(X_train_w, y_train_w)

wrong_acc = model_wrong.score(X_holdout_w, y_holdout_w)
print("Leaky holdout accuracy:", round(wrong_acc, 3))

Wow!! This result looks impressive, but remember: **the labels are random**.

The apparent signal was manufactured by allowing holdout cases to influence which variables were selected.

#### Correct it yourself! (split first, then select using train only)

Rewrite the workflow so that:

1. the split happens first;
2. the feature selector sees only `X_train` and `y_train`;
3. the exact same selected features are applied to `X_holdout`;
4. the classifier is fitted only on the transformed training data;
5. performance is evaluated on the holdout set.

In [ ]:
# Write your corrected workflow here

### In summary: 

The train/holdout rule is always unchanged:

> **The selection procedure is learned from training data only.**


**Wrong**

```text
ALL DATA
   |
   v
select features using X and y
   |
   v
train/holdout split
   |
   v
evaluate
```

**Correct**

```text
ALL DATA
   |
   v
train/holdout split
   |
   +------ TRAIN ------> select features ------> fit model
   |
   +------ HOLDOUT  ------> keep same features ------> evaluate
```

With thousands of candidate variables, some will correlate with a random target purely by chance. If the holdout labels participate in that search, the holdout set is no longer independent.

## 6. Putting everything together

Fitted transformations, scaling, imputation, and feature selection look like different topics, but methodologically they are the same because they all **learn parameters or decisions from the data**.

That means that they must be learned from the training set and only applied to the holdout set. That is, something like this should be performed:


```text
RAW DATA
   |
   v
TRAIN / HOLDOUT SPLIT
   |
   +------ TRAIN
   |         +--> learn fitted transformations   
   |         +--> learn scaling
   |         +--> learn imputation
   |         +--> learn feature selection
   |         +--> fit model
   |
   +------ HOLDOUT
             +--> apply training transformations
             +--> apply training scaling
             +--> apply training imputation
             +--> keep training-selected features
             +--> evaluate
```

This can make your code long, but `Pipeline` from `scikit-learn` can help you with this very easily.

A pipeline is more than a convenient syntax. It encodes:

$$
\boxed{\text{fit preprocessing on train} \rightarrow \text{apply to holdout}}
$$


Let's do this ourselves by building a final pipeline one step at a time.

**Question**: Before that, what should come next?

1. missingness;
2. scaling;
3. feature selection;
4. model.

YOUR RESPONSE HERE AS MARKDOWN

In [ ]:
from sklearn.pipeline import Pipeline
feature_cols = [
    "age", "income", "credit_score", "transactions",
    "debt_ratio", "noise_1", "noise_2"
]
X = df[feature_cols]
y = df["target"]


X_train, X_holdout, y_train, y_holdout = None, None, None, None # HERE COMES CODE: Create a training/holdout partition, with test_size=.30, random_state=42, stratify=y
pipe = None # HERE COMES CODE: Create a pipeline with the decided order above

# HERE COMES CODE: Fit pipeline to the training set
# HERE COMES CODE: Show the accuracy of the fitted pipeline on the holdout

## 7. Summary

For every feature-wrangling step, ask:

$$
\boxed{\text{Does this step learn anything from the observed data?}}
$$

If yes:

$$
\boxed{\text{learn it from train only}}
$$

Then apply the learned rule to both train and holdout.

| Step | What is learned? |
|---|---|
| Mean / median imputation | training mean / median |
| Missingness indicator | which variables were missing during training |
| Standardization | training mean and SD |
| Min-max scaling | training min and max |
| Robust scaling | training median and IQR |
| Quantile clipping / binning | training thresholds / cut points |
| Power transform | fitted transformation parameter |
| Feature selection | which predictors survive |
| Model fitting | model parameters |

$$
\boxed{\text{Split first. Wrangle second.}}
$$

## 📝 8. Exercises

**8.1 Find the leakage**

### A
```python
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_train, X_holdout, y_train, y_holdout = train_test_split(X_scaled, y)
```

YOUR RESPONSE HERE AS MARKDOWN

### B
```python
X_train, X_holdout, y_train, y_holdout = train_test_split(X, y)
X_train["income"] = np.log1p(X_train["income"])
X_holdout["income"]  = np.log1p(X_holdout["income"])
```

YOUR RESPONSE HERE AS MARKDOWN


### C
```python
X_train, X_holdout, y_train, y_holdout = train_test_split(X, y)
median = X_train["age"].median()
X_train["age"] = X_train["age"].fillna(median)
X_holdout["age"]  = X_holdout["age"].fillna(median)
```

YOUR RESPONSE HERE AS MARKDOWN

### D
```python
selector = SelectKBest(k=10)
X_selected = selector.fit_transform(X, y)
X_train, X_holdout, y_train, y_holdout = train_test_split(X_selected, y)
```

YOUR RESPONSE HERE AS MARKDOWN